### Runnable 객체

**실행 가능 객체**

+ `Runnable`은 단일 입력에 대해 출력을 생성할 수 있는 모든 객체를 말한다. 

+ `ChatPromptTemplate` (프롬프트), `ChatGoogleGenerativeAI` (모델), `StrOutputParser` (파서) 등이 모두 `Runnable`이다.

**체인 구성**

+ `|` (파이프) 연산자를 사용하여 여러 `Runnable` 객체를 연결하여 하나의 복합적인 **체인**을 만들 수 있다. 

+ 연결된 체인 자체도 `Runnable` 객체로 취급된다.

**다양한 실행 방식**

+ `invoke()`, `stream()`, `batch()`, `ainvoke()`와 같은 메서드를 통해 단일 실행, 스트리밍, 병렬 처리 등 다양한 방식으로 작업을 수행할 수 있다.

### LCEL(LangChain Expression Language)

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

gemini_api_key = os.getenv("GEMINI_API_KEY")

In [2]:
from langchain_core.prompts import PromptTemplate

template = "{language} 할 수 있어?"

prompt = PromptTemplate.from_template(template)
prompt

PromptTemplate(input_variables=['language'], input_types={}, partial_variables={}, template='{language} 할 수 있어?')

In [18]:
from langchain_google_genai import ChatGoogleGenerativeAI

model = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.1, # 창의성 (0.0 ~ 2.0)
    google_api_key=gemini_api_key
)


In [6]:
chain = prompt | model

In [7]:
input = {"language": "한국말"}

chain.invoke(input)

AIMessage(content='네, 한국말 할 수 있습니다. 무엇을 도와드릴까요?', additional_kwargs={}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': []}, id='run--8acd3373-3f8b-4c19-a010-d05eb9316623-0', usage_metadata={'input_tokens': 7, 'output_tokens': 45, 'total_tokens': 52, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 31}})

### 스트리밍 출력

In [19]:
template = "{topic}에 대해 쉽게 설명해줘?"

prompt = PromptTemplate.from_template(template)

In [20]:
input = {"topic": "LangChain"}

In [21]:
chain = prompt | model

In [23]:
response = chain.stream(input)

In [25]:
next(response)

AIMessageChunk(content='LangChain은 쉽게 말해 **"똑똑한 뇌(LLM)에게 손발과 기억력, 그리고 계획 능력을 달아주는 프레임워크"** 라고 생각하시면 됩니다.\n\n조금 더 자세히', additional_kwargs={}, response_metadata={'safety_ratings': []}, id='run--48ba9809-4dd5-427e-a70a-27778a37a4a3', usage_metadata={'input_tokens': 10, 'output_tokens': 1861, 'total_tokens': 1871, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 1813}})

In [13]:
for token in response:
    print(token.content, end="", flush=True)

LangChain에 대해 쉽게 설명해 드릴게요! 🤖🛠️

---

**LangChain, 쉽게 말해볼까요?**

LLM(Large Language Model, 거대 언어 모델)은 정말 똑똑한 '뇌'라고 생각하시면 돼요. 질문에 답하고, 글을 쓰고, 요약하는 등 언어와 관련된 일은 기가 막히게 잘하죠.

그런데 이 똑똑한 뇌도 혼자서는 할 수 없는 일들이 있어요. 예를 들면:
*   "지금 날씨가 어때?" 라고 물었을 때, LLM은 날씨 정보를 '알고' 있는 게 아니라, '날씨 정보가 필요하다'는 걸 이해할 뿐이에요. 실제로 인터넷에 접속해서 날씨를 검색할 수는 없죠.
*   "내 이메일에서 중요한 내용만 요약해 줘" 라고 했을 때, LLM은 내 이메일 계정에 접속해서 내용을 가져올 수 없어요.
*   "이 문서 내용을 바탕으로 보고서를 써줘" 라고 했을 때, LLM은 문서를 읽고 요약하는 건 잘하지만, 그 내용을 바탕으로 복잡한 구조의 보고서를 '단계별로' 작성하는 건 어려울 수 있어요.

**여기서 LangChain이 등장합니다!**

LangChain은 이 똑똑한 LLM '뇌'에 **'손발'을 달아주고, '기억력'을 주고, '복잡한 일 처리 능력'을 부여해서 훨씬 더 유용하고 강력한 인공지능 애플리케이션을 만들 수 있게 도와주는 '도구 상자' 또는 '비서' 같은 존재예요.**

---

**LangChain이 하는 일 (핵심 기능):**

1.  **도구 (Tools) 🛠️:**
    *   LLM이 외부 세상과 상호작용할 수 있게 해주는 기능이에요.
    *   **예시:** 인터넷 검색 도구, 계산기 도구, 데이터베이스 조회 도구, 특정 API 호출 도구 등.
    *   이제 LLM은 "지금 날씨가 어때?"라는 질문을 받으면, LangChain이 제공하는 '인터넷 검색 도구'를 사용해서 실제 날씨 정보를 찾아올 수 있게 되는 거죠!

2.  **체인 (Chains) 🔗:**
    *   여러 단계를 연결해서 하나의 큰 작업을 수행하게 해주는 기